# Monitorización de Calidad del Agua en Embalses con ndvi2gif

Este notebook demuestra cómo usar **ndvi2gif** para:

1. Cargar un shapefile de embalses de la CHG (Confederación Hidrográfica del Guadalquivir)
2. Generar composites mensuales de Sentinel-2 para detección de agua
3. Calcular máscaras de agua mensuales mediante umbrales en índices espectrales
4. Analizar índices de calidad del agua específicos de Sentinel-2
5. Visualizar la evolución temporal de la superficie de agua y parámetros de calidad

### Índices utilizados

| Índice | Descripción | Bandas S2 | Uso |
|--------|-------------|-----------|-----|
| NDWI | Normalized Difference Water Index | Green / NIR | Detección de agua |
| MNDWI | Modified NDWI | Green / SWIR1 | Detección de agua mejorada |
| AWEI | Automated Water Extraction Index | Blue, Green, NIR, SWIR1, SWIR2 | Extracción robusta de agua |
| AWEInsh | AWEI no shadow | Green, NIR, SWIR1, SWIR2 | Detección sin sombras |
| NDCI | Normalized Difference Chlorophyll Index | Red Edge 1 / Red | Clorofila-a (exclusivo S2) |
| WI2015 | Water Index 2015 | NIR, SWIR1, SWIR2 | Índice de agua adicional |


## 1. Instalación y configuración

In [ ]:
# pip install ndvi2gif  # Descomentar si no está instalado

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="geemap.conversion")

import os
import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

from ndvi2gif import NdviSeasonality, TimeSeriesAnalyzer

print("Modulos importados correctamente")

# Autenticar e inicializar Earth Engine
# ee.Authenticate()  # Solo la primera vez
ee.Initialize()
print("Earth Engine inicializado")

## 2. Carga y exploración del shapefile de embalses

In [ ]:
# Ruta al shapefile de la CHG
shp_path = 'CHG.ZonasProtegidas.CaptacionesEmbalses.shp'

# Cargar con geopandas
gdf = gpd.read_file(shp_path)

print(f"Sistema de coordenadas: {gdf.crs}")
print(f"Numero de embalses: {len(gdf)}")
print(f"\nColumnas disponibles:")
print(gdf.columns.tolist())
print(f"\nPrimeras filas:")
gdf.head(10)

In [ ]:
# Visualizar la distribucion geografica
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
gdf.plot(ax=ax, color='steelblue', edgecolor='navy', alpha=0.7)
ax.set_title('Embalses CHG - Zona de Captacion', fontsize=14)
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
plt.tight_layout()
plt.show()

# Informacion de los embalses
print(f"\nExtent total: {gdf.total_bounds}")
print(f"\nEstadisticas de area (m2):")
gdf['area_m2'] = gdf.geometry.to_crs('EPSG:32630').area
print(gdf['area_m2'].describe())

In [ ]:
# Seleccion del embalse a analizar
# Opcion A: usar todos los embalses como ROI (union)
# Opcion B: filtrar por nombre o ID un embalse especifico

# Ver columnas que pueden contener el nombre
# Sustituir 'NOMBRE' por el nombre real de la columna que aparece arriba
nombre_col = gdf.columns[0]  # ajustar segun la columna de nombre disponible
print("Lista de embalses disponibles:")
for i, row in gdf.iterrows():
    print(f"  [{i}] {row[nombre_col]} | Area: {row.get('area_m2', 0)/1e6:.2f} km2")

In [ ]:
# Seleccionar el embalse de mayor superficie para el analisis
# (cambiar el criterio o usar filtros por nombre segun convenga)
idx_mayor = gdf['area_m2'].idxmax()
embalse_seleccionado = gdf.loc[[idx_mayor]]

print(f"Embalse seleccionado: {embalse_seleccionado.iloc[0][nombre_col]}")
print(f"Area: {embalse_seleccionado.iloc[0]['area_m2']/1e6:.2f} km2")
print(f"Coordenadas: {embalse_seleccionado.total_bounds}")

# Guardar como shapefile temporal para usarlo como ROI
embalse_path = '/tmp/embalse_seleccionado.shp'
embalse_seleccionado.to_file(embalse_path)
print(f"\nShapefile guardado en: {embalse_path}")

# Tambien podemos crear un buffer para capturar las orillas y zona de transicion
embalse_buffer = embalse_seleccionado.to_crs('EPSG:32630').buffer(500).to_crs('EPSG:4326')
embalse_buffer_gdf = gpd.GeoDataFrame(geometry=embalse_buffer, crs='EPSG:4326')
embalse_buffer_path = '/tmp/embalse_buffer.shp'
embalse_buffer_gdf.to_file(embalse_buffer_path)
print(f"Shapefile con buffer (500m) guardado en: {embalse_buffer_path}")

## 3. Deteccion de agua: composites mensuales con NDWI y MNDWI

Usamos `periods=12` para obtener composites mensuales con Sentinel-2.

El **MNDWI** es generalmente preferible al NDWI clasico en zonas con
presencia de suelo desnudo, edificaciones y vegetacion acuatica, ya que
usa SWIR en lugar de NIR, lo que suprime mejor las areas no acuaticas.

In [ ]:
# --- Composite mensual NDWI (Gao 1996) ---
# Umbral tipico para agua: NDWI > 0

s2_ndwi = NdviSeasonality(
    roi=embalse_buffer_path,   # Usamos el buffer para capturar la zona de transicion
    periods=12,                # Mensual
    start_year=2020,
    end_year=2025,
    sat='S2',
    key='median',              # Mediana para reducir ruido en agua
    index='ndwi'
)

composite_ndwi = s2_ndwi.get_year_composite()
print("Bandas NDWI mensuales:", composite_ndwi.first().bandNames().getInfo())

In [ ]:
# --- Composite mensual MNDWI (Xu 2006) ---
# Mas robusto que NDWI en zonas con urbanizacion y vegetacion acuatica

s2_mndwi = NdviSeasonality(
    roi=embalse_buffer_path,
    periods=12,
    start_year=2020,
    end_year=2025,
    sat='S2',
    key='median',
    index='mndwi'
)

composite_mndwi = s2_mndwi.get_year_composite()
print("Bandas MNDWI mensuales:", composite_mndwi.first().bandNames().getInfo())

In [ ]:
# --- Composite mensual AWEI (Feyisa 2014) ---
# Algoritmo de extraccion de agua automatico, muy robusto frente a sombras

s2_awei = NdviSeasonality(
    roi=embalse_buffer_path,
    periods=12,
    start_year=2020,
    end_year=2025,
    sat='S2',
    key='median',
    index='aweinsh'            # AWEInsh: sin penalizacion por sombras, adecuado para embalses
)

composite_awei = s2_awei.get_year_composite()
print("Bandas AWEInsh mensuales:", composite_awei.first().bandNames().getInfo())

## 4. Visualizacion de los composites de deteccion de agua

In [ ]:
Map1 = geemap.Map()
Map1.centerObject(composite_mndwi, zoom=11)

# Parametros de visualizacion para indices de agua
# Valores positivos = agua, negativos = tierra
water_viz = {
    'bands': ['july'],
    'min': -0.3,
    'max': 0.5,
    'palette': ['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#74add1', '#313695']
}

# Comparativa entre anos: mediana de todos los anos para julio
mndwi_julio = composite_mndwi.select('july')
ndwi_julio  = composite_ndwi.select('july')
awei_julio  = composite_awei.select('july')

Map1.addLayer(ndwi_julio.median(),  water_viz, 'NDWI - Julio (mediana interanual)')
Map1.addLayer(mndwi_julio.median(), water_viz, 'MNDWI - Julio (mediana interanual)')
Map1.addLayer(awei_julio.median(),
              {'bands': ['july'], 'min': -2000, 'max': 2000,
               'palette': ['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#74add1', '#313695']},
              'AWEInsh - Julio (mediana interanual)')

Map1

## 5. Mascaras binarias de agua

Aplicamos un umbral para generar mascaras binarias:
- **MNDWI > 0**: agua (valor 1)
- **MNDWI <= 0**: no-agua (valor 0)

Este umbral funciona bien en la mayoria de embalses ibericos. En zonas
muy turbias puede requerir ajuste (p.ej. MNDWI > 0.1).

In [ ]:
# Umbral para mascara binaria
WATER_THRESHOLD = 0.0  # Ajustar si es necesario (tipicamente 0.0 a 0.1)

MESES = ['january', 'february', 'march', 'april', 'may', 'june',
         'july', 'august', 'september', 'october', 'november', 'december']

def crear_mascara_agua(composite, threshold=WATER_THRESHOLD):
    """Convierte un composite de indice de agua en mascara binaria.
    
    Devuelve una ImageCollection donde cada imagen tiene bandas
    mensuales con valor 1 (agua) o 0 (no-agua).
    """
    def mask_image(image):
        return image.gt(threshold).rename(image.bandNames())
    return composite.map(mask_image)

mascaras_agua = crear_mascara_agua(composite_mndwi)

# Comprobar resultado
primera_img = mascaras_agua.first()
print("Bandas de mascara:", primera_img.bandNames().getInfo())
print("Valores unicos (deberia ser 0 y 1):")
stats = primera_img.select('july').reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=primera_img.geometry(),
    scale=20,
    maxPixels=1e8
).getInfo()
print(stats)

In [ ]:
Map2 = geemap.Map()
Map2.centerObject(composite_mndwi, zoom=11)

mascara_viz = {
    'bands': ['july'],
    'min': 0,
    'max': 1,
    'palette': ['#f5f5f5', '#2166ac']  # blanco = tierra, azul = agua
}

# Mascara de agua para diferentes meses con la mediana interanual
for mes in ['january', 'april', 'july', 'october']:
    mascara_mes = mascaras_agua.select(mes).median()
    Map2.addLayer(
        mascara_mes,
        {'bands': [mes], 'min': 0, 'max': 1, 'palette': ['#f5f5f5', '#2166ac']},
        f'Mascara agua - {mes.capitalize()}'
    )

Map2

## 6. Calculo de superficie de agua mensual

Calculamos la superficie inundada en km2 para cada mes y cada año.

In [ ]:
def calcular_superficie_agua(composite_mascara, roi_geom, pixel_area_m2=400):
    """
    Calcula el area de agua (km2) para cada banda mensual de cada imagen.
    
    Parameters
    ----------
    composite_mascara : ee.ImageCollection
        Coleccion de mascaras binarias (0/1) con bandas mensuales.
    roi_geom : ee.Geometry
        Geometria del area de interes.
    pixel_area_m2 : float
        Area de pixel en m2 (S2 = 20m -> 400 m2 para MNDWI).
    
    Returns
    -------
    pd.DataFrame con columnas [año, mes, area_km2]
    """
    resultados = []
    imagenes = composite_mascara.toList(composite_mascara.size())
    n_imgs = composite_mascara.size().getInfo()
    
    for i in range(n_imgs):
        img = ee.Image(imagenes.get(i))
        year = img.date().get('year').getInfo()
        
        for mes in MESES:
            try:
                banda = img.select(mes)
                # Sumar pixeles de agua y multiplicar por area de pixel
                suma = banda.reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi_geom,
                    scale=20,
                    maxPixels=1e9
                ).get(mes).getInfo()
                
                if suma is not None:
                    area_km2 = suma * pixel_area_m2 / 1e6
                    resultados.append({'year': year, 'month': mes, 'area_km2': area_km2})
            except Exception:
                pass  # Banda no disponible para este periodo
    
    return pd.DataFrame(resultados)

# Obtener geometria del ROI desde Earth Engine
roi_ee = ee.FeatureCollection(embalse_buffer_path).geometry()

print("Calculando superficie de agua mensual... (puede tardar unos minutos)")
df_area = calcular_superficie_agua(mascaras_agua, roi_ee)
print(f"Calculados {len(df_area)} registros")
df_area.head(15)

In [ ]:
# Ordenar meses correctamente para el grafico
orden_meses = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
               'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
df_area['month_num'] = df_area['month'].map(orden_meses)
df_area = df_area.sort_values(['year', 'month_num'])

# --- Grafico de evolucion mensual por año ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Series temporales por año
ax1 = axes[0]
colores = plt.cm.viridis(np.linspace(0, 1, df_area['year'].nunique()))
for (año, grupo), color in zip(df_area.groupby('year'), colores):
    ax1.plot(grupo['month_num'], grupo['area_km2'],
             marker='o', linewidth=2, label=str(año), color=color)
ax1.set_xlabel('Mes', fontsize=12)
ax1.set_ylabel('Superficie de agua (km²)', fontsize=12)
ax1.set_title('Evolucion mensual de la superficie de agua (MNDWI > 0)', fontsize=13)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                      'Jul','Ago','Sep','Oct','Nov','Dic'])
ax1.legend(title='Año', bbox_to_anchor=(1.01, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Media mensual con desviacion estandar
ax2 = axes[1]
media_mensual = df_area.groupby('month_num')['area_km2'].agg(['mean', 'std']).reset_index()
ax2.bar(media_mensual['month_num'], media_mensual['mean'],
        color='steelblue', alpha=0.7, label='Media interanual')
ax2.errorbar(media_mensual['month_num'], media_mensual['mean'],
             yerr=media_mensual['std'], fmt='none', color='navy', capsize=4)
ax2.set_xlabel('Mes', fontsize=12)
ax2.set_ylabel('Superficie de agua (km²)', fontsize=12)
ax2.set_title('Media mensual interanual de superficie de agua', fontsize=13)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                      'Jul','Ago','Sep','Oct','Nov','Dic'])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('superficie_agua_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: superficie_agua_mensual.png")

## 7. Indices de calidad del agua con Sentinel-2

### NDCI - Normalized Difference Chlorophyll Index

El NDCI usa la banda **Red Edge 1 (B5, 705 nm)** exclusiva de Sentinel-2,
optimizada para detectar clorofila-a y cianobacterias en aguas continentales.

**Formula:** `(B5 - B4) / (B5 + B4)`

- Valores altos (> 0.1): alta concentracion de clorofila, posibles floraciones algales
- Valores bajos (< 0): agua con baja productividad biologica

**Referencia:** Mishra & Mishra (2012), Remote Sensing of Environment

In [ ]:
# NDCI mensual - solo disponible para Sentinel-2 (usa Red Edge B5)
s2_ndci = NdviSeasonality(
    roi=embalse_buffer_path,
    periods=12,
    start_year=2020,
    end_year=2025,
    sat='S2',
    key='median',
    index='ndci'       # Normalized Difference Chlorophyll Index - exclusivo S2
)

composite_ndci = s2_ndci.get_year_composite()
print("Bandas NDCI mensuales:", composite_ndci.first().bandNames().getInfo())

In [ ]:
# WI2015 - Water Index 2015 como indice adicional de agua
s2_wi2015 = NdviSeasonality(
    roi=embalse_buffer_path,
    periods=12,
    start_year=2020,
    end_year=2025,
    sat='S2',
    key='median',
    index='wi2015'
)

composite_wi2015 = s2_wi2015.get_year_composite()
print("Bandas WI2015 mensuales:", composite_wi2015.first().bandNames().getInfo())

## 8. Calidad del agua: NDCI solo sobre pixeles de agua

Para analizar correctamente la calidad del agua, aplicamos la mascara
de agua (MNDWI > 0) antes de interpretar el NDCI.

In [ ]:
def aplicar_mascara_agua_a_indice(composite_indice, composite_mascara, threshold=0.0):
    """
    Enmascara un composite de indice de calidad de agua
    usando solo los pixeles identificados como agua.
    """
    def mask_per_image(args):
        img_indice  = ee.Image(ee.List(args).get(0))
        img_mascara = ee.Image(ee.List(args).get(1))
        mascara_binaria = img_mascara.gt(threshold)
        return img_indice.updateMask(mascara_binaria)
    
    lista_indice  = composite_indice.toList(composite_indice.size())
    lista_mascara = composite_mascara.toList(composite_mascara.size())
    
    # Asegurar que ambas colecciones tienen el mismo tamanyo
    n = composite_indice.size().min(composite_mascara.size())
    indices = ee.List.sequence(0, n.subtract(1))
    
    masked = indices.map(
        lambda i: ee.Image(lista_indice.get(i))
                    .updateMask(ee.Image(lista_mascara.get(i)).gt(threshold))
    )
    return ee.ImageCollection(masked)

composite_ndci_masked = aplicar_mascara_agua_a_indice(composite_ndci, composite_mndwi)
print("NDCI enmascarado sobre agua calculado")

In [ ]:
# Visualizacion comparativa: NDWI, MNDWI, NDCI
Map3 = geemap.Map()
Map3.centerObject(composite_ndci, zoom=11)

# Palette para indices de agua (NDWI/MNDWI)
water_palette = ['#d73027', '#fc8d59', '#fee090', '#e0f3f8', '#74add1', '#313695']

# Palette para clorofila (NDCI): rojo = baja, verde = alta concentracion
chl_palette = ['#053061', '#2166ac', '#4393c3', '#92c5de', '#d1e5f0',
               '#f7f7f7', '#fddbc7', '#f4a582', '#d6604d', '#b2182b', '#67001f']

# Mediana interanual para julio (mes de maxima radiacion, mayor riesgo de floracion)
mes_analisis = 'july'

Map3.addLayer(
    composite_mndwi.select(mes_analisis).median(),
    {'bands': [mes_analisis], 'min': -0.3, 'max': 0.6, 'palette': water_palette},
    f'MNDWI - {mes_analisis.capitalize()}'
)
Map3.addLayer(
    composite_ndci_masked.select(mes_analisis).median(),
    {'bands': [mes_analisis], 'min': -0.2, 'max': 0.3, 'palette': chl_palette},
    f'NDCI (clorofila-a) - {mes_analisis.capitalize()}'
)
Map3.addLayer(
    composite_wi2015.select(mes_analisis).median(),
    {'bands': [mes_analisis], 'min': -0.5, 'max': 0.5, 'palette': water_palette},
    f'WI2015 - {mes_analisis.capitalize()}'
)

Map3

## 9. Analisis temporal de la calidad del agua

Extraemos la media mensual del NDCI sobre los pixeles de agua
para monitorizar la evolucion de la clorofila-a a lo largo del tiempo.

In [ ]:
def extraer_estadisticas_mensuales(composite, roi_geom, banda_base, scale=20):
    """
    Extrae media, mediana y desviacion tipica de un composite mensual
    para cada imagen (año) de la coleccion.
    
    Returns
    -------
    pd.DataFrame con columnas [year, month, mean, median, std]
    """
    resultados = []
    imagenes = composite.toList(composite.size())
    n_imgs = composite.size().getInfo()
    
    reducers = (
        ee.Reducer.mean()
        .combine(ee.Reducer.median(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    )
    
    for i in range(n_imgs):
        img = ee.Image(imagenes.get(i))
        year = img.date().get('year').getInfo()
        
        for mes in MESES:
            try:
                banda = img.select(mes)
                stats = banda.reduceRegion(
                    reducer=reducers,
                    geometry=roi_geom,
                    scale=scale,
                    maxPixels=1e9
                ).getInfo()
                
                resultados.append({
                    'year': year,
                    'month': mes,
                    'mean': stats.get(f'{mes}_mean'),
                    'median': stats.get(f'{mes}_median'),
                    'std': stats.get(f'{mes}_stdDev')
                })
            except Exception:
                pass
    
    df = pd.DataFrame(resultados)
    df['month_num'] = df['month'].map(orden_meses)
    return df.sort_values(['year', 'month_num'])

print("Extrayendo estadisticas mensuales del NDCI sobre agua...")
df_ndci = extraer_estadisticas_mensuales(composite_ndci_masked, roi_ee, 'ndci')
print(f"Extraidos {len(df_ndci)} registros")
df_ndci.head(12)

In [ ]:
# --- Grafico de calidad del agua: NDCI mensual ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: NDCI mensual por año (proxy de clorofila-a)
ax1 = axes[0]
colores = plt.cm.plasma(np.linspace(0.1, 0.9, df_ndci['year'].nunique()))
for (año, grupo), color in zip(df_ndci.groupby('year'), colores):
    ax1.plot(grupo['month_num'], grupo['mean'],
             marker='o', linewidth=2, label=str(año), color=color)

# Zona de alerta (NDCI > 0.1 -> posible floracion algal)
ax1.axhline(0.1, color='orange', linestyle='--', linewidth=1.5,
            label='Umbral floracion algal (NDCI=0.1)')
ax1.axhline(0.2, color='red', linestyle='--', linewidth=1.5,
            label='Alerta bloom (NDCI=0.2)')
ax1.fill_between(range(1, 13), 0.1, 0.5, alpha=0.08, color='orange')
ax1.fill_between(range(1, 13), 0.2, 0.5, alpha=0.08, color='red')

ax1.set_xlabel('Mes', fontsize=12)
ax1.set_ylabel('NDCI medio (clorofila-a proxy)', fontsize=12)
ax1.set_title('Evolucion mensual del NDCI - Indice de Clorofila-a (Sentinel-2)', fontsize=13)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                      'Jul','Ago','Sep','Oct','Nov','Dic'])
ax1.legend(title='Año', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Heatmap anyo x mes del NDCI
ax2 = axes[1]
pivot = df_ndci.pivot(index='year', columns='month_num', values='mean')
pivot.columns = ['Ene','Feb','Mar','Abr','May','Jun',
                  'Jul','Ago','Sep','Oct','Nov','Dic']
im = ax2.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                vmin=-0.1, vmax=0.3)
ax2.set_xticks(range(12))
ax2.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                      'Jul','Ago','Sep','Oct','Nov','Dic'])
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels(pivot.index)
ax2.set_title('Heatmap NDCI (clorofila-a) - Rojo = alta concentracion', fontsize=13)
plt.colorbar(im, ax=ax2, label='NDCI')

plt.tight_layout()
plt.savefig('ndci_calidad_agua.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: ndci_calidad_agua.png")

## 10. Analisis con TimeSeriesAnalyzer

Usamos el modulo `TimeSeriesAnalyzer` para detectar tendencias
en la evolucion de la clorofila-a (NDCI) a lo largo del periodo.

In [ ]:
# Analisis de tendencias con TimeSeriesAnalyzer
ts_ndci = TimeSeriesAnalyzer(
    processor=s2_ndci  # Reutilizamos el procesador ya inicializado
)

# Extraer serie temporal
ts_data = ts_ndci.extract_time_series(
    geometry=roi_ee,
    reducer='mean',
    scale=20
)
print("Serie temporal extraida:")
print(ts_data.head())

In [ ]:
# Estadisticas de tendencia
trend = ts_ndci.compute_trend_statistics()
print("=== Tendencia en NDCI (clorofila-a) ===")
print(f"Test Mann-Kendall: tau={trend.get('tau', 'N/A'):.3f}, p-value={trend.get('p_value', 'N/A'):.4f}")
print(f"Pendiente Sen: {trend.get('sen_slope', 'N/A'):.6f} por periodo")
if trend.get('p_value', 1) < 0.05:
    dir_trend = 'creciente (deterioro de calidad)' if trend.get('sen_slope', 0) > 0 else 'decreciente (mejora de calidad)'
    print(f"Tendencia significativa: {dir_trend}")
else:
    print("Tendencia no significativa (p > 0.05)")

In [ ]:
# Visualizacion completa de la serie temporal
ts_ndci.plot_comprehensive_analysis(title='NDCI - Evolucion de Clorofila-a en Embalse')

## 11. Resumen comparativo de indices de agua

Tabla resumen con los indices disponibles en ndvi2gif para monitoreo de embalses.

In [ ]:
# Resumen: medias anuales de los distintos indices
print("=" * 65)
print("RESUMEN DE INDICES DE AGUA DISPONIBLES EN ndvi2gif para S2")
print("=" * 65)
print()
indices_info = [
    ("NDWI",    "ndwi",    "Deteccion agua (clasico)",               "Green/NIR",          "Todos"),
    ("MNDWI",   "mndwi",   "Deteccion agua mejorada",                "Green/SWIR1",         "Todos"),
    ("AWEInsh", "aweinsh", "Extraccion agua auto (sin sombra)",      "Blue/Green/NIR/SWIR", "Todos"),
    ("AWEI",    "awei",    "Extraccion agua auto (con sombra)",      "Blue/Green/NIR/SWIR", "Todos"),
    ("WI2015",  "wi2015",  "Indice de agua 2015",                    "NIR/SWIR1/SWIR2",     "Todos"),
    ("NDCI",    "ndci",    "Clorofila-a / cianobacterias (calidad)", "Red Edge1/Red",       "Solo S2"),
    ("NDMI",    "ndmi",    "Humedad (vegetacion acuatica/orilla)",   "NIR/SWIR1",           "Todos"),
]
print(f"{'Indice':<10} {'Clave':<10} {'Descripcion':<42} {'Bandas':<22} {'Sensores'}")
print("-" * 100)
for nombre, clave, desc, bandas, sensores in indices_info:
    print(f"{nombre:<10} {clave:<10} {desc:<42} {bandas:<22} {sensores}")

print()
print("Para indices S3 (OLCI, 300m): oci, tsi, cdom, turbidity, spm, kd490, floating_algae")

## 12. Exportar resultados a Google Drive / GeoTIFF

Si queremos exportar las mascaras de agua para procesarlas en GIS:

In [ ]:
# Exportar la mascara de agua media interanual mensual a Google Drive
# (Descomentar para ejecutar la exportacion)

# Mediana interanual de la mascara de agua MNDWI
mascara_media = mascaras_agua.median()

# task = ee.batch.Export.image.toDrive(
#     image=mascara_media,
#     description='Mascara_agua_mensual_MNDWI',
#     folder='ndvi2gif_outputs',
#     fileNamePrefix='mascara_agua_mndwi_2020_2024',
#     region=roi_ee,
#     scale=20,
#     crs='EPSG:4326',
#     maxPixels=1e10
# )
# task.start()
# print(f"Tarea de exportacion enviada. Estado: {task.status()}")

# Exportar NDCI enmascarado
# ndci_media = composite_ndci_masked.median()
# task2 = ee.batch.Export.image.toDrive(
#     image=ndci_media,
#     description='NDCI_clorofila_mensual',
#     folder='ndvi2gif_outputs',
#     fileNamePrefix='ndci_mensual_2020_2024',
#     region=roi_ee,
#     scale=20,
#     crs='EPSG:4326',
#     maxPixels=1e10
# )
# task2.start()
# print(f"Tarea NDCI enviada. Estado: {task2.status()}")

print("Descomenta las lineas anteriores para exportar a Google Drive")

## Notas y referencias

### Indices utilizados

- **NDWI**: Gao, B. (1996). NDWI — A normalized difference water index. *Remote Sensing of Environment*, 58(3), 257–266.
- **MNDWI**: Xu, H. (2006). Modification of normalised difference water index (NDWI). *International Journal of Remote Sensing*, 27(14), 3025–3033.
- **AWEI**: Feyisa, G.L. et al. (2014). Automated Water Extraction Index: A new technique for surface water mapping. *Remote Sensing of Environment*, 140, 23–35.
- **NDCI**: Mishra, S. & Mishra, D.R. (2012). Normalized difference chlorophyll index. *Remote Sensing of Environment*, 117, 394–406.

### Umbrales recomendados para deteccion de agua

| Indice | Umbral agua | Notas |
|--------|------------|-------|
| NDWI   | > 0.0      | Puede sobreestimar agua en areas urbanas |
| MNDWI  | > 0.0      | Mas preciso en entornos heterogeneos |
| AWEInsh| > 0        | Robusto, preferible en zonas de montana |

### Interpretacion del NDCI

| NDCI | Estado del agua |
|------|----------------|
| < 0  | Agua limpia / baja clorofila |
| 0 – 0.1 | Concentracion moderada |
| 0.1 – 0.2 | Concentracion elevada (vigilar) |
| > 0.2 | Posible floracion algal (alerta) |
